# Olist B2B Revenue Funnel - business analysis
SQL controls, descriptive analysis, reproducible findings and recommendations. No predictive model.

In [1]:
from pathlib import Path
import sqlite3, json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == '04_Python' else Path.cwd()
con = sqlite3.connect(ROOT / '03_SQL/analysis.sqlite')

In [2]:
sql = (ROOT / '03_SQL/business_analysis.sql').read_text()
import re
for name, query in re.findall(r'-- name: (\w+)\n(.*?);', sql, re.S):
    table = pd.read_sql_query(query, con)
    print('\n'+name+'\n'+table.head(12).to_string(index=False))


portfolio_kpis
 leads  closed_deals  conversion_rate  active_sellers      gmv  gmv_per_converted_seller  close_days
  8000           842          0.10525             376 664858.0                789.617577   49.160558

funnel_stages
               stage  sellers
                 MQL     8000
              Closed      842
First delivered sale      376
       Repeat seller      295

channel_quality
           origin  leads  closed_deals  conversion_rate  active_sellers  activation_rate       gmv  gmv_per_lead  gmv_per_converted  close_days
   direct_traffic    499            56         0.112224              31         0.553571  21852.90     43.793387         390.230357   31.767716
          display    118             6         0.050847               2         0.333333    923.00      7.822034         153.833333   10.921914
            email    493            15         0.030426               6         0.400000   8484.99     17.210933         565.666000   52.773204
   organic_search   2296

In [3]:
findings = json.loads((ROOT / '08_Documentation/findings.json').read_text())
print('FINDINGS\n'+'\n'.join(findings['findings']))
print('\nRECOMMENDATIONS\n'+'\n'.join(findings['recommendations']))
con.close()

FINDINGS
8,000 MQLs produced 842 closed deals (10.53% conversion).
376 converted sellers had delivered orders purchased after closing (44.66% activation).
Attributed item GMV was BRL 664,858.00; this is marketplace merchandise value, not Olist net revenue.
Top 38 active sellers (ceiling of 10%) contributed 64.82% of attributed GMV.
295 sellers generated at least two post-close delivered orders.
unknown had the highest observed GMV per lead among channels with at least 50 MQLs: BRL 181.96.

RECOMMENDATIONS
Use observed GMV per lead together with activation to shortlist channel experiments; acquire spend data before claiming ROI.
Prioritize onboarding for converted sellers without post-close delivered orders. Separate recent wins from fully observed cohorts.
Inspect the largest seller accounts individually because concentration makes channel results sensitive to a few merchants.
Obtain assignment histories for lost leads before measuring SDR or sales-representative conversion rates.


In [4]:
print((ROOT / '08_Documentation/statistical_test.json').read_text())

{
  "test": "Mann-Whitney U, descriptive association, exploratory",
  "n_a": 2296,
  "n_b": 1586,
  "mean_a": 0.11803135888501742,
  "mean_b": 0.12295081967213115,
  "p_value": 0.6430052606780108,
  "caution": "Unadjusted for confounding and multiple comparisons. No causal interpretation."
}


## Interpretation limits
- Marketplace GMV is item price, not Olist commission revenue. Freight and platform take rate are excluded.
- SDR, sales representative, business segment and lead type exist only on won deals. Their conversion rates cannot be computed without assignments for lost leads.
- Activation is the first observed post-close delivered order; active seller and first-order stages cannot be separately identified.
- Recent acquisitions have shorter observation windows. No causal channel comparison, channel ROI, CAC or long-term LTV is claimed.
- Only converted sellers linking to marketplace orders can be monetized. Sellers without matches remain in the activation denominator.
- Negative lead-to-close durations are excluded from closing-speed averages, but those deal records remain in conversion and attribution counts.